In [2]:
import logging
import os

import h5py
import hist
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

hep.style.use(hep.style.ROOT)
logging.basicConfig(level=logging.INFO)

plot_dir = os.path.join(os.getcwd(), "validate_jets_plots")
os.makedirs(plot_dir, exist_ok=True)

FILES = {
    "tt": "/storage/af/user/tsievert/topNet/fjTag_testing.h5",
    "qcd": "/storage/af/user/tsievert/topNet/qcd_4j_Alltesting.h5",
    # "4t": "",
}
LOADED_FILES = {label: h5py.File(filepath) for label, filepath in FILES.items()}

N_TOPS = 2
N_JETS = 10
N_FJETS = 3

PLOT_STYLINGS = {
    # Jets
    'pt': (100, 20, 350), 'eta': (30, -5, 5), 'phi': (30, -np.pi - 0.2, np.pi + 0.2), 'btag': (10, 0, 1), 'mass': (100, 0, 150),

    # BoostedJets
    'fj_pt': (100, 100, 600), 'fj_eta': (30, -5, 5), 'fj_phi': (30, -np.pi - 0.2, np.pi + 0.2), 'fj_Ttag': (10, 0, 1), 'fj_Wtag': (10, 0, 1), 
    'fj_mass': (100, 0, 250), 'fj_sdmass': (100, 00, 250), 'fj_tau21': (10, 0, 1), 'fj_tau32': (10, 0, 1), 'fj_ncharged': (20, 0, 50)
}

for input, binning in PLOT_STYLINGS.items():
    if input.startswith('fj_'): file_add = 'Boosted'; njets = N_FJETS
    else: file_add = ''; njets = N_JETS

    for jetcol in range(njets):
        fig, ax = plt.subplots()

        for file_label, file in LOADED_FILES.items():
            full_input = file[f"INPUTS/{file_add}Jets/{input}"][:]
            mask = file[f"INPUTS/{file_add}Jets/MASK"][:]
            input_hist = hist.Hist(
                hist.axis.Boolean(name="var")
                if input.endswith('tag') else
                hist.axis.Regular(binning[0], binning[1], binning[2], name="var", growth=False)
            ).fill(var=full_input[:, jetcol][mask[:, jetcol]])
            hep.histplot(input_hist, label=file_label, ax=ax, density=True)

        ax.set_xlabel(input)
        ax.set_ylabel('Density of N events')
        ax.legend(loc="upper right", fontsize=10)
        plt.tight_layout()
        jetcol_dir = os.path.join(plot_dir, f"{file_add}Jets_{jetcol}")
        os.makedirs(jetcol_dir, exist_ok=True)
        fig.savefig(os.path.join(jetcol_dir, f"{input}.png"))
        # fig.savefig(os.path.join(plot_dir, f"{input}.pdf"))
        plt.close()


In [ ]:
for file in LOADED_FILES.values(): file.close()